# Практика · DETR: детекція як передбачення множини

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі дані зошит генерує формулами, ваги не
> завантажуються. Досить `torch`, `torchvision`, `scipy`, `numpy` і `matplotlib`.

> ⏱ Зошит навчає **сім** маленьких мереж: три DETR, три anchor-free детектори
> й одну DETR без зіставлення. Заміряно окремим прогоном: **348 секунд**,
> тобто близько шести хвилин на чотирьох ядрах без відеокарти — приблизно
> 56 секунд на одну DETR і 28 на anchor-free.

Що зробимо:

1. напишемо **свій GIoU** і звіримо з `torchvision.ops.generalized_box_iou`;
2. напишемо **свій угорський алгоритм** і звіримо з `scipy.optimize.linear_sum_assignment`;
3. порахуємо руками, де жадібне зіставлення програє оптимальному, і на скільки;
4. складемо **матрицю вартості** з трьох частин і покрутимо ваги;
5. покажемо, що наївна втрата за індексом залежить від порядку, а зіставлена — ні;
6. зберемо **власну маленьку DETR** і навчимо її на сценах 64×64 у три зерна;
7. навчимо anchor-free детектор із теми 26 на тому самому бюджеті й порівняємо криві;
8. поміряємо, скільки запитів кажуть «порожньо» і як вони поділили сцену;
9. покажемо, що **без однозначного зіставлення** запити злипаються в одну точку;
10. порахуємо **дублікати до NMS** в обох моделей;
11. перевіримо, чи розвʼязує DETR сцену, на якій NMS губить справжній предмет.

In [ ]:
import math
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import box_iou, nms, generalized_box_iou, sigmoid_focal_loss
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми йдуть в іншому порядку
torch.set_num_threads(1)

print("torch      ", torch.__version__)
print("numpy      ", np.__version__)
print("потоків    ", torch.get_num_threads())

## 1 · Датасет: ті самі сцени 64×64

Датасет не міняємо — це той самий генератор, що в темах 23, 25 і 26. Полотно
64 × 64, від одного до трьох предметів трьох класів, рамка береться **з маски**,
тобто істинна за побудовою. Так числа цієї теми можна класти поруч із числами
попередніх.

In [ ]:
SIZE = 64                                  # сторона полотна в пікселях
CLASS_NAMES = ["коло", "квадрат", "трикутник"]
CLASS_COUNT = 3
GRID = 8                                   # карта ознак 8×8
STRIDE = SIZE / GRID                       # крок карти: 8 пікселів на клітинку


def shape_mask(kind, center_x, center_y, radius):
    '''Маска однієї фігури на полотні 64×64.'''
    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    if kind == 0:                                   # коло
        return (xs - center_x) ** 2 + (ys - center_y) ** 2 <= radius * radius
    if kind == 1:                                   # квадрат
        return (np.abs(xs - center_x) <= radius) & (np.abs(ys - center_y) <= radius)
    # трикутник: ширина росте згори вниз
    return ((ys - center_y + radius >= 0) & (ys - center_y <= radius)
            & (np.abs(xs - center_x) <= (ys - center_y + radius) / 2.0))


def make_scene(rng, min_objects=1, max_objects=4, min_radius=6, max_radius=11):
    '''Одна сцена: картинка, рамки з масок, мітки класів.'''
    image = np.zeros((SIZE, SIZE), np.float32)
    boxes, labels = [], []
    for _ in range(int(rng.integers(min_objects, max_objects))):
        for _attempt in range(40):
            radius = int(rng.integers(min_radius, max_radius))
            center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
            center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
            kind = int(rng.integers(0, 3))
            mask = shape_mask(kind, center_x, center_y, radius)
            ys, xs = np.nonzero(mask)
            box = [float(xs.min()), float(ys.min()),
                   float(xs.max() + 1), float(ys.max() + 1)]

            # не даємо предметам злипатись більше ніж на 45 % площі нового
            too_close = False
            for previous in boxes:
                overlap_w = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                overlap_h = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if overlap_w * overlap_h > 0.45 * (box[2] - box[0]) * (box[3] - box[1]):
                    too_close = True
                    break
            if too_close:
                continue

            image[mask] = 1.0
            boxes.append(box)
            labels.append(kind)
            break
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return image, np.array(boxes, np.float32), np.array(labels, np.int64)


def make_dataset(seed, count):
    rng = np.random.default_rng(seed)
    return [make_scene(rng) for _ in range(count)]


started = time.time()
train_set = make_dataset(42, 240)
test_set = make_dataset(7, 120)

train_images = torch.from_numpy(np.stack([s[0] for s in train_set])[:, None])
test_images = torch.from_numpy(np.stack([s[0] for s in test_set])[:, None])
train_targets = [(torch.from_numpy(s[1]), torch.from_numpy(s[2])) for s in train_set]
ground_truth = [(torch.from_numpy(s[1]), torch.from_numpy(s[2])) for s in test_set]
true_box_count = sum(len(g[0]) for g in ground_truth)

print("згенеровано за %.2f с" % (time.time() - started))
print("навчальних сцен %d, перевірних %d" % (len(train_set), len(test_set)))
print("істинних рамок у перевірному наборі: %d" % true_box_count)
print("предметів на сцену в середньому: %.2f" % (true_box_count / len(test_set)))

Одразу порахуємо **середню істинну рамку** перевірного набору — вона знадобиться
далі як точка відліку: якби запити DETR не спеціалізувались, усі вони цілились би
саме сюди.

In [ ]:
all_centers_x, all_centers_y, all_sides = [], [], []
for _image, boxes, _labels in test_set:
    for box in boxes:
        all_centers_x.append((box[0] + box[2]) / 2)
        all_centers_y.append((box[1] + box[3]) / 2)
        all_sides.append(math.sqrt((box[2] - box[0]) * (box[3] - box[1])))

print("середній центр істинних рамок:  x = %.1f, y = %.1f"
      % (np.mean(all_centers_x), np.mean(all_centers_y)))
print("середній бік істинної рамки:    %.1f px" % np.mean(all_sides))
print("розкид центрів по всіх рамках:  x %.1f, y %.1f"
      % (np.std(all_centers_x), np.std(all_centers_y)))

## 2 · Свій GIoU проти бібліотечного

`IoU` мовчить, коли рамки не перетинаються: він дорівнює нулю і для сусідньої
рамки, і для рамки на іншому кінці картинки. `GIoU` додає штраф за порожнє місце
в найменшому прямокутнику, який накриває обидві рамки, — і тому продовжує
відрізняти «близько» від «далеко».

In [ ]:
def giou_by_hand(a, b):
    '''Матриця GIoU між наборами рамок a (N×4) і b (M×4), формат xyxy.'''
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])

    # перетин: правий-нижній мінімум мінус лівий-верхній максимум
    lt = torch.max(a[:, None, :2], b[None, :, :2])
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b[None, :] - inter
    iou = inter / union

    # накривний прямокутник: найменший, у який влазять обидві рамки
    lt_c = torch.min(a[:, None, :2], b[None, :, :2])
    rb_c = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh_c = (rb_c - lt_c).clamp(min=0)
    enclosing = wh_c[:, :, 0] * wh_c[:, :, 1]

    # штраф — частка накривного прямокутника, яку не займає жодна рамка
    return iou - (enclosing - union) / enclosing


probe_a = torch.tensor([[10., 10., 30., 30.], [20., 20., 40., 40.], [0., 0., 6., 6.]])
probe_b = torch.tensor([[12., 12., 32., 32.], [50., 50., 60., 60.]])
mine = giou_by_hand(probe_a, probe_b)
library = generalized_box_iou(probe_a, probe_b)

print("наш GIoU:")
print(mine.numpy().round(6))
print("бібліотечний:")
print(library.numpy().round(6))
gap = float((mine - library).abs().max())
assert np.allclose(mine.numpy(), library.numpy()), "GIoU розійшовся!"
print("\n✅ збігається, максимальна розбіжність %.2e" % gap)

А тепер головне: **що GIoU каже там, де IoU мовчить**. Беремо дві рамки 10 × 10
й розсуваємо їх усе далі. `IoU` в усіх рядках однаковий нуль, а `GIoU` монотонно
спадає — тобто далі дає градієнту напрямок «наближайся».

In [ ]:
probe_box = torch.tensor([[0., 0., 10., 10.]])
print("відстань   IoU      GIoU")
for distance in (12, 20, 30, 40):
    other = torch.tensor([[float(distance), float(distance),
                           float(distance + 10), float(distance + 10)]])
    print("  %2d px   %.4f   %+.4f"
          % (distance,
             float(box_iou(probe_box, other)[0, 0]),
             float(giou_by_hand(probe_box, other)[0, 0])))

## 3 · Свій угорський алгоритм проти `scipy`

Задача про призначення: кожній істині призначити рівно одне передбачення так,
щоб сумарна вартість була найменшою. Реалізуємо метод Мункреса — той самий, що
всередині `scipy.optimize.linear_sum_assignment`. Код нижче довгий, але кожен
його блок відповідає одному кроку з лекції: віднімання по рядках, віднімання по
стовпцях, пошук незалежних нулів.

In [ ]:
def hungarian_by_hand(cost):
    '''Оптимальне зіставлення рядків до стовпців. Повертає (rows, cols).'''
    c = np.array(cost, dtype=float)
    n, m = c.shape

    # прямокутну таблицю доповнюємо до квадратної дорогими клітинками:
    # алгоритм вимагає, щоб кожен рядок і кожен стовпець дістав пару
    size = max(n, m)
    big = c.max() + 1.0 if c.size else 1.0
    square = np.full((size, size), big)
    square[:n, :m] = c

    # крок 1 і 2: віднімаємо мінімум рядка, потім мінімум стовпця
    square = square - square.min(axis=1, keepdims=True)
    square = square - square.min(axis=0, keepdims=True)

    # крок 3: жадібно позначаємо «зірками» нулі, що не ділять рядка чи стовпця
    starred = np.zeros((size, size), bool)
    row_used = np.zeros(size, bool)
    col_used = np.zeros(size, bool)
    for i in range(size):
        for j in range(size):
            if square[i, j] == 0 and not row_used[i] and not col_used[j]:
                starred[i, j] = True
                row_used[i] = col_used[j] = True

    while True:
        covered_cols = starred.any(axis=0)
        if covered_cols.sum() >= size:          # зірок стільки, скільки рядків
            break
        primed = np.zeros((size, size), bool)
        covered_rows = np.zeros(size, bool)
        while True:
            # шукаємо непокритий нуль
            spot = None
            for i in range(size):
                if covered_rows[i]:
                    continue
                for j in range(size):
                    if covered_cols[j]:
                        continue
                    if abs(square[i, j]) < 1e-12:
                        spot = (i, j)
                        break
                if spot:
                    break
            if spot is None:
                # непокритих нулів немає — робимо нові, зсуваючи непокриту частину
                free = square[~covered_rows][:, ~covered_cols].min()
                square[~covered_rows] -= free
                square[:, covered_cols] += free
                continue
            i, j = spot
            primed[i, j] = True
            if starred[i].any():
                covered_rows[i] = True
                covered_cols[starred[i].argmax()] = False
            else:
                # нарощуємо чергу «штрих — зірка» і міняємо їх місцями
                path = [(i, j)]
                while True:
                    col = path[-1][1]
                    if starred[:, col].any():
                        row = int(starred[:, col].argmax())
                        path.append((row, col))
                        path.append((row, int(primed[row].argmax())))
                    else:
                        break
                for r, cc in path:
                    starred[r, cc] = not starred[r, cc]
                break

    rows, cols = np.nonzero(starred)
    keep = (rows < n) & (cols < m)              # відкидаємо доповнені клітинки
    return rows[keep], cols[keep]


rng_check = np.random.default_rng(0)
mismatches = 0
for _trial in range(300):
    n, m = int(rng_check.integers(2, 8)), int(rng_check.integers(1, 5))
    matrix = rng_check.random((n, m)) * 3
    our_rows, our_cols = hungarian_by_hand(matrix)
    lib_rows, lib_cols = linear_sum_assignment(matrix)
    if abs(matrix[our_rows, our_cols].sum() - matrix[lib_rows, lib_cols].sum()) > 1e-9:
        mismatches += 1

assert mismatches == 0, "власний угорський алгоритм розійшовся зі scipy!"
print("✅ 300 випадкових матриць, розбіжностей: %d" % mismatches)

## 4 · Головний замір теми: жадібно проти оптимально

Спершу приклад руками, три передбачення й дві істини. Жадібний спосіб бере
найдешевшу клітинку таблиці, викреслює її рядок і стовпець, повторює.

In [ ]:
def greedy_assign(cost):
    '''Жадібне зіставлення: найдешевша клітинка, потім викреслюємо рядок і стовпець.'''
    c = np.array(cost, dtype=float).copy()
    n, m = c.shape
    rows, cols = [], []
    for _ in range(min(n, m)):
        flat = int(np.argmin(c))
        i, j = flat // m, flat % m
        if not np.isfinite(c[i, j]):
            break
        rows.append(i)
        cols.append(j)
        c[i, :] = np.inf                # рядок і стовпець більше не розглядаємо
        c[:, j] = np.inf
    return np.array(rows), np.array(cols)


hand = np.array([[0.30, 0.40],
                 [0.35, 0.90],
                 [1.20, 1.30]])
print("матриця вартості (менше — краще):")
for index, row in enumerate(hand):
    print("   П%d   %.2f   %.2f" % (index + 1, row[0], row[1]))

greedy_rows, greedy_cols = greedy_assign(hand)
best_rows, best_cols = linear_sum_assignment(hand)
print()
print("жадібно:   " + ", ".join("П%d→І%d" % (r + 1, c + 1)
                                for r, c in zip(greedy_rows, greedy_cols))
      + "   сума %.2f" % hand[greedy_rows, greedy_cols].sum())
print("угорський: " + ", ".join("П%d→І%d" % (r + 1, c + 1)
                                for r, c in zip(best_rows, best_cols))
      + "   сума %.2f" % hand[best_rows, best_cols].sum())
print("жадібний переплатив %.2f"
      % (hand[greedy_rows, greedy_cols].sum() - hand[best_rows, best_cols].sum()))

Один приклад нічого не доводить. Проженемо два способи по **двох тисячах**
випадкових матриць розміру 10 × M — саме такої форми, яку дає DETR із десятьма
запитами й одним-трьома предметами в сцені.

In [ ]:
rng_many = np.random.default_rng(5)
different, overpays = 0, []
for _trial in range(2000):
    truth_count = int(rng_many.integers(1, 4))
    cost = rng_many.random((10, truth_count)) * 4 - 1
    g_rows, g_cols = greedy_assign(cost)
    b_rows, b_cols = linear_sum_assignment(cost)
    if set(zip(g_rows.tolist(), g_cols.tolist())) != set(zip(b_rows.tolist(), b_cols.tolist())):
        different += 1
    overpays.append(cost[g_rows, g_cols].sum() - cost[b_rows, b_cols].sum())

print("розійшлись у %d випадках із 2000 (%.1f %%)" % (different, different / 20))
print("середня переплата жадібного: %.4f" % float(np.mean(overpays)))
print("найгірша переплата:          %.4f" % float(np.max(overpays)))

## 5 · Вартість зіставлення з трьох частин

Вартість пари «передбачення — істина» складається з трьох доданків: класу,
`L1` по нормованих координатах і `1 − GIoU`. Канонічні ваги зі статті — 1, 5 і 2.

Візьмімо приклад, у якому доданки **сперечаються**. Друга істина — квадрат.
Передбачення П2 має ідеально точну рамку, але майже не вірить у клас (0.15).
Передбачення П3 стоїть на піксель гірше по кожній координаті, зате впевнене
(0.70). Кому дістанеться друга істина — вирішують ваги.

In [ ]:
COST_CLASS, COST_L1, COST_GIOU = 1.0, 5.0, 2.0


def cost_matrix(probabilities, predicted_boxes, truth_labels, truth_boxes,
                w_class=COST_CLASS, w_l1=COST_L1, w_giou=COST_GIOU):
    '''Матриця вартості N×M: клас + L1 по нормованих координатах + (1 − GIoU).'''
    # мінус: чим упевненіше передбачення в потрібному класі, тим дешевша пара
    class_part = -probabilities[:, truth_labels]
    # координати ділимо на сторону сцени, щоб L1 не залежав від розміру картинки
    l1_part = torch.cdist(predicted_boxes / SIZE, truth_boxes / SIZE, p=1)
    giou_part = 1 - giou_by_hand(predicted_boxes, truth_boxes)
    return w_class * class_part + w_l1 * l1_part + w_giou * giou_part


example_truth = torch.tensor([[10., 10., 30., 30.], [34., 20., 50., 36.]])
example_labels = torch.tensor([0, 1])                   # коло і квадрат
example_boxes = torch.tensor([[12., 12., 32., 32.],
                              [34., 20., 50., 36.],
                              [33., 19., 49., 35.]])
example_prob = torch.tensor([[0.80, 0.10, 0.05, 0.05],
                             [0.60, 0.15, 0.10, 0.15],
                             [0.10, 0.70, 0.10, 0.10]])

base = cost_matrix(example_prob, example_boxes, example_labels, example_truth)
print("матриця вартості при канонічних вагах 1 / 5 / 2:")
print("          І1 (коло)   І2 (квадрат)")
for index in range(3):
    print("   П%d       %7.3f        %7.3f" % (index + 1, base[index, 0], base[index, 1]))

print()
print("ваги (клас/L1/GIoU)   зіставлення            сума")
for w_class, w_l1, w_giou in ((1, 5, 2), (3, 5, 2), (1, 10, 4)):
    matrix = cost_matrix(example_prob, example_boxes, example_labels, example_truth,
                         w_class, w_l1, w_giou).numpy()
    rows, cols = linear_sum_assignment(matrix)
    pairs = ", ".join("П%d→І%d" % (r + 1, c + 1) for r, c in zip(rows, cols))
    print("    %2d / %2d / %2d        %-20s %7.3f"
          % (w_class, w_l1, w_giou, pairs, matrix[rows, cols].sum()))

Читається так: при канонічних вагах геометрія переважає й другу істину забирає
П2. Варто підняти вагу класу втричі — і вона переходить до П3. А от однакове
збільшення **обох** геометричних ваг (10 і 4 замість 5 і 2) не міняє нічого:
важить не величина ваг, а їхнє співвідношення.

## 6 · Множина не має порядку

Тепер покажемо, навіщо взагалі потрібне зіставлення. Візьмемо чотири
передбачення й дві істини й порахуємо втрату двома способами: наївно **попарно
за індексом** (перше з першим, друге з другим) і **після зіставлення**. Потім
переберемо всі 24 перестановки чотирьох передбачень.

In [ ]:
order_truth = torch.tensor([[8., 10., 26., 28.], [34., 30., 52., 48.]])
order_labels = torch.tensor([0, 1])
order_boxes = torch.tensor([[10., 12., 28., 30.],
                            [33., 29., 51., 47.],
                            [40.,  6., 54., 20.],
                            [ 6., 40., 20., 54.]])
order_prob = torch.tensor([[0.75, 0.10], [0.08, 0.70], [0.30, 0.25], [0.20, 0.30]])


def pair_loss(prediction_index, truth_index):
    '''Втрата однієї пари: клас через −log p, геометрія та сама, що у вартості.'''
    probability = max(1e-6, float(order_prob[prediction_index, order_labels[truth_index]]))
    box = order_boxes[prediction_index:prediction_index + 1]
    truth = order_truth[truth_index:truth_index + 1]
    l1 = float(torch.cdist(box / SIZE, truth / SIZE, p=1)[0, 0])
    giou = float(giou_by_hand(box, truth)[0, 0])
    return -math.log(probability) + 5 * l1 + 2 * (1 - giou)


def loss_by_index(order):
    '''Наївна втрата: k-те передбачення порівнюємо з k-ю істиною.'''
    return sum(pair_loss(order[k], k) for k in range(len(order_truth)))


matching_cost = np.zeros((4, 2))
for i in range(4):
    for j in range(2):
        matching_cost[i, j] = pair_loss(i, j)
matched_rows, matched_cols = linear_sum_assignment(matching_cost)
loss_after_matching = matching_cost[matched_rows, matched_cols].sum()

import itertools
values = [loss_by_index(list(order)) for order in itertools.permutations(range(4))]

print("втрата за індексом, порядок 1 2 3 4:  %.2f" % loss_by_index([0, 1, 2, 3]))
print("втрата за індексом, порядок 2 1 3 4:  %.2f" % loss_by_index([1, 0, 2, 3]))
print("розмах по всіх 24 перестановках:      %.2f … %.2f (у %.2f раза)"
      % (min(values), max(values), max(values) / min(values)))
print("втрата після зіставлення:             %.2f — і вона одна на всі 24"
      % loss_after_matching)

## 7 · Модель: маленька DETR

Тіло — те саме, що в темах 23, 25 і 26. Далі карта ознак 8 × 8 розгортається в
послідовність із 64 векторів, до кожного додається навчуваний позиційний вектор,
енкодер дає їм подивитись одне на одного, а декодер приводить `N = 10` запитів і
питає карту, що для них є.

In [ ]:
QUERIES = 10                               # скільки передбачень мережа віддає завжди
D_MODEL = 64                               # ширина векторів у трансформері


class Body(nn.Module):
    '''Те саме тіло, що в темах 23, 25 і 26: 64×64 → карта 8×8 на 64 канали.'''

    def __init__(self):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2))

        self.net = nn.Sequential(
            block(1, 16), block(16, 32), block(32, 64),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())

    def forward(self, x):
        return self.net(x)


class TinyDetr(nn.Module):
    '''Маленька DETR: тіло, позиційне кодування, енкодер, декодер, дві голови.'''

    def __init__(self, queries=QUERIES, layers=2):
        super().__init__()
        self.body = Body()
        # позиційний вектор на кожну з 64 клітинок карти: без нього енкодер
        # не знає, звідки взявся кожен вектор послідовності
        self.position = nn.Parameter(torch.zeros(GRID * GRID, D_MODEL))
        nn.init.normal_(self.position, std=0.02)
        # запити мусять бути РІЗНИМИ: увага еквіваріантна щодо перестановки,
        # тож однакові запити дали б однакові відповіді
        self.query = nn.Parameter(torch.zeros(queries, D_MODEL))
        nn.init.normal_(self.query, std=1.0)

        encoder_layer = nn.TransformerEncoderLayer(
            D_MODEL, nhead=4, dim_feedforward=128, dropout=0.0,
            batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, layers,
                                             enable_nested_tensor=False)
        decoder_layer = nn.TransformerDecoderLayer(
            D_MODEL, nhead=4, dim_feedforward=128, dropout=0.0,
            batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, layers)

        self.class_head = nn.Linear(D_MODEL, CLASS_COUNT + 1)   # +1 — «порожньо»
        self.box_head = nn.Sequential(nn.Linear(D_MODEL, 64), nn.ReLU(),
                                      nn.Linear(64, 4))

    def forward(self, x):
        count = x.shape[0]
        features = self.body(x).flatten(2).permute(0, 2, 1)      # (B, 64, D)
        memory = self.encoder(features + self.position[None])
        queries = self.query[None].expand(count, -1, -1)
        decoded = self.decoder(queries, memory)
        logits = self.class_head(decoded)
        # сигмоїда тримає рамку всередині картинки за побудовою
        boxes = torch.sigmoid(self.box_head(decoded))            # cx, cy, w, h у [0,1]
        return logits, boxes


def cxcywh_to_xyxy(boxes):
    '''Із центра й розмірів у два кути — формат, з яким працюють box_iou і GIoU.'''
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)


probe = TinyDetr()
body_parameters = sum(p.numel() for p in probe.body.parameters())
print("параметрів у DETR усього:      %d" % sum(p.numel() for p in probe.parameters()))
print("   з них у тілі:               %d" % body_parameters)
print("   в енкодері:                 %d" % sum(p.numel() for p in probe.encoder.parameters()))
print("   у декодері:                 %d" % sum(p.numel() for p in probe.decoder.parameters()))
print("   у самих запитах:            %d (%d запитів × %d чисел)"
      % (probe.query.numel(), QUERIES, D_MODEL))
print("   у позиційному кодуванні:    %d" % probe.position.numel())
print("   у двох головах:             %d"
      % (sum(p.numel() for p in probe.class_head.parameters())
         + sum(p.numel() for p in probe.box_head.parameters())))

## 8 · Втрата DETR

Спершу зіставлення — **без градієнта**, це просто список пар. Потім та сама
вартість, але як функція втрати: клас через крос-ентропію з меншою вагою для
«порожньо», геометрія ділена на кількість зіставлених пар.

Аргумент `use_matching=False` вмикає наївний варіант, у якому кожна істина тягне
до себе **всі** запити одразу. Він знадобиться в розділі 13.

In [ ]:
NO_OBJECT_WEIGHT = 0.1                       # «порожньо» важить удесятеро менше
CLASS_WEIGHTS = torch.tensor([1.0] * CLASS_COUNT + [NO_OBJECT_WEIGHT])


def detr_loss(logits, boxes, targets, use_matching=True):
    '''Втрата DETR: спершу зіставлення, потім та сама вартість як функція втрати.'''
    count, queries = logits.shape[0], logits.shape[1]
    probabilities = torch.softmax(logits, dim=-1)
    box_xyxy = cxcywh_to_xyxy(boxes) * SIZE

    # усі запити за замовчуванням мають казати «порожньо»
    target_class = torch.full((count, queries), CLASS_COUNT, dtype=torch.long)
    l1_sum = logits.sum() * 0                # нульовий тензор із правильним графом
    giou_sum = logits.sum() * 0
    matched = 0

    for index, (truth_boxes, truth_labels) in enumerate(targets):
        if len(truth_boxes) == 0:
            continue
        with torch.no_grad():                # зіставлення градієнта не потребує
            cost = cost_matrix(probabilities[index], box_xyxy[index].detach(),
                               truth_labels, truth_boxes)
            if use_matching:
                rows, cols = linear_sum_assignment(cost.numpy())
            else:
                # без зіставлення: кожна істина тягне ВСІ запити до себе
                rows = np.repeat(np.arange(queries), len(truth_boxes))
                cols = np.tile(np.arange(len(truth_boxes)), queries)
        target_class[index, rows] = truth_labels[cols]
        l1_sum = l1_sum + F.l1_loss(box_xyxy[index][rows] / SIZE,
                                    truth_boxes[cols] / SIZE, reduction="sum")
        pair_giou = giou_by_hand(box_xyxy[index][rows], truth_boxes[cols])
        giou_sum = giou_sum + (1 - torch.diagonal(pair_giou)).sum()
        matched += len(rows)

    matched = max(1, matched)
    class_loss = F.cross_entropy(logits.reshape(-1, CLASS_COUNT + 1),
                                 target_class.reshape(-1), weight=CLASS_WEIGHTS)
    # ділимо на кількість пар, а не на N: інакше сцена з трьома предметами
    # давала б утричі більший градієнт, ніж сцена з одним
    return class_loss + (COST_L1 * l1_sum + COST_GIOU * giou_sum) / matched


print("вага класу «порожньо» у крос-ентропії: %.1f" % NO_OBJECT_WEIGHT)
print("ваги геометричних доданків: L1 = %.0f, GIoU = %.0f" % (COST_L1, COST_GIOU))
print("зіставлення рахується під torch.no_grad() — градієнт крізь нього не йде")

## 9 · Оцінювання: mAP так само, як у темах 23 і 26

Беремо ті самі функції, щоб числа були порівнянні. Єдина відмінність — у DETR
**немає NMS**: її передбачення йдуть в оцінку як є.

In [ ]:
def greedy_match(boxes, scores, truth_boxes, iou_threshold):
    '''Жадібне зіставлення за спаданням оцінки — те саме, що в темі 23.'''
    order = torch.argsort(scores, descending=True)
    taken = [False] * len(truth_boxes)
    hits = torch.zeros(len(order))
    if len(truth_boxes) and len(order):
        overlaps = box_iou(boxes[order], truth_boxes)
        for position in range(len(order)):
            best_value, best_index = -1.0, -1
            for truth_index in range(len(truth_boxes)):
                if taken[truth_index]:
                    continue
                if overlaps[position, truth_index].item() > best_value:
                    best_value = overlaps[position, truth_index].item()
                    best_index = truth_index
            if best_index >= 0 and best_value >= iou_threshold:
                taken[best_index] = True
                hits[position] = 1.0
    return scores[order], hits


def average_precision(scores, hits, truth_count):
    '''AP як площа під огинальною кривої точність-повнота.'''
    order = torch.argsort(scores, descending=True)
    ordered_hits = hits[order]
    running_hits = torch.cumsum(ordered_hits, 0)
    running_misses = torch.cumsum(1 - ordered_hits, 0)
    precision = running_hits / (running_hits + running_misses)
    recall = running_hits / truth_count

    envelope = precision.clone()
    for i in range(len(envelope) - 2, -1, -1):
        envelope[i] = max(envelope[i].item(), envelope[i + 1].item())

    area, previous_recall = 0.0, 0.0
    for i in range(len(envelope)):
        area += (recall[i].item() - previous_recall) * envelope[i].item()
        previous_recall = recall[i].item()
    return area


def mean_average_precision(predictions, iou_threshold=0.5, nms_threshold=None,
                           score_floor=1e-3):
    '''mAP при одному порозі IoU. nms_threshold=None — NMS не застосовуємо.'''
    values = []
    for class_index in range(CLASS_COUNT):
        score_parts, hit_parts, truth_count = [], [], 0
        for (boxes, scores, labels), (truth_boxes, truth_labels) in zip(predictions,
                                                                        ground_truth):
            chosen = (scores >= score_floor) & (labels == class_index)
            picked_boxes, picked_scores = boxes[chosen], scores[chosen]
            if nms_threshold is not None and len(picked_boxes):
                keep = nms(picked_boxes, picked_scores, nms_threshold)
                picked_boxes, picked_scores = picked_boxes[keep], picked_scores[keep]
            this_truth = truth_boxes[truth_labels == class_index]
            truth_count += len(this_truth)
            ordered, hits = greedy_match(picked_boxes, picked_scores, this_truth,
                                         iou_threshold)
            score_parts.append(ordered)
            hit_parts.append(hits)
        values.append(average_precision(torch.cat(score_parts), torch.cat(hit_parts),
                                        max(1, truth_count)))
    return float(np.mean(values)), values


@torch.no_grad()
def predict_detr(model, images):
    '''Оцінка запиту — найбільша ймовірність серед справжніх класів.'''
    model.eval()
    logits, boxes = model(images)
    probabilities = torch.softmax(logits, dim=-1)
    xyxy = cxcywh_to_xyxy(boxes) * SIZE
    results = []
    for index in range(images.shape[0]):
        score, class_id = probabilities[index, :, :CLASS_COUNT].max(dim=1)
        results.append((xyxy[index], score, class_id))
    return results


print("оцінювання готове: mAP рахується так само, як у темах 23 і 26,")
print("але для DETR nms_threshold лишається None — NMS у її конвеєрі немає")

## 10 · Навчання DETR: три зерна

36 епох, AdamW зі швидкістю навчання 3·10⁻³ і косинусним спаданням, партії по
32 сцени. Кожні три епохи міряємо `mAP@0.5` — з цих точок далі складеться крива
збіжності.

In [ ]:
def train_detr(seed, epochs=36, learning_rate=3e-3, use_matching=True, track=None):
    torch.manual_seed(seed)
    model = TinyDetr()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                                  weight_decay=1e-4)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    curve, spent = [], 0.0
    for epoch in range(epochs):
        started = time.time()
        model.train()
        order = torch.randperm(len(train_images))
        for start in range(0, len(train_images), 32):
            batch = order[start:start + 32]
            logits, boxes = model(train_images[batch])
            loss = detr_loss(logits, boxes,
                             [train_targets[i] for i in batch.tolist()],
                             use_matching=use_matching)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        schedule.step()
        spent += time.time() - started
        if track and (epoch + 1) % track == 0:
            value, _per_class = mean_average_precision(predict_detr(model, test_images))
            curve.append((epoch + 1, value))
    return model, spent, curve


detr_curves, detr_scores, detr_models = [], [], {}
for seed in (0, 1, 2):
    model, spent, curve = train_detr(seed, track=3)
    value, _per_class = mean_average_precision(predict_detr(model, test_images))
    detr_curves.append(curve)
    detr_scores.append(value)
    detr_models[seed] = model
    print("  DETR, зерно %d: %5.1f с   mAP@0.5 = %.4f" % (seed, spent, value))

print("  середнє %.4f, розкид %.4f"
      % (float(np.mean(detr_scores)), max(detr_scores) - min(detr_scores)))

## 11 · Суперник: anchor-free детектор із теми 26

Щоб порівняння було чесним, беремо той самий детектор, що переміг у темі 26:
те саме тіло, focal loss, чотири відстані до країв замість якорів. Навчаємо його
на **тому самому бюджеті** — 36 епох, три зерна.

In [ ]:
PRIOR = 0.01                                    # бажана ймовірність предмета на старті
BIAS_INIT = -math.log((1 - PRIOR) / PRIOR)


def build_points():
    '''Центр кожної клітинки карти 8×8 у пікселях вхідного зображення.'''
    out = []
    for row in range(GRID):
        for col in range(GRID):
            out.append([(col + 0.5) * STRIDE, (row + 0.5) * STRIDE])
    return torch.tensor(out, dtype=torch.float32)


POINTS = build_points()


def free_targets(boxes, labels):
    '''Ціль anchor-free: позиція позитивна, якщо вона всередині рамки.'''
    total = POINTS.shape[0]
    positive = torch.zeros(total, dtype=torch.bool)
    class_id = torch.full((total,), -1, dtype=torch.long)
    distances = torch.zeros(total, 4)
    if len(boxes) == 0:
        return positive, class_id, distances

    truth = torch.from_numpy(boxes)
    areas = (truth[:, 2] - truth[:, 0]) * (truth[:, 3] - truth[:, 1])
    point_x, point_y = POINTS[:, 0:1], POINTS[:, 1:2]
    left, top = point_x - truth[:, 0], point_y - truth[:, 1]
    right, bottom = truth[:, 2] - point_x, truth[:, 3] - point_y

    inside = (left > 0) & (top > 0) & (right > 0) & (bottom > 0)
    # серед рамок, що містять точку, беремо найменшу за площею
    area_or_infinity = torch.where(inside, areas.expand_as(inside),
                                   torch.full_like(inside, float("inf"),
                                                   dtype=torch.float32))
    smallest, chosen = area_or_infinity.min(dim=1)
    positive = torch.isfinite(smallest)

    index = torch.arange(total)
    stacked = torch.stack([left[index, chosen], top[index, chosen],
                           right[index, chosen], bottom[index, chosen]], dim=1)
    distances[positive] = stacked[positive] / STRIDE
    class_id[positive] = torch.from_numpy(labels)[chosen[positive]]
    return positive, class_id, distances


class FreeDetector(nn.Module):
    '''Anchor-free голова з теми 26: класи й чотири відстані на позицію.'''

    def __init__(self):
        super().__init__()
        self.body = Body()
        self.classifier = nn.Conv2d(64, CLASS_COUNT, 3, padding=1)
        self.regressor = nn.Conv2d(64, 4, 3, padding=1)
        nn.init.constant_(self.classifier.bias, BIAS_INIT)

    def forward(self, x):
        features = self.body(x)
        count = x.shape[0]
        logits = self.classifier(features).permute(0, 2, 3, 1).reshape(count, -1,
                                                                       CLASS_COUNT)
        distances = F.relu(self.regressor(features)).permute(0, 2, 3, 1).reshape(count,
                                                                                 -1, 4)
        return logits, distances


def pack_free(dataset):
    total = POINTS.shape[0]
    positive = torch.zeros(len(dataset), total, dtype=torch.bool)
    class_target = torch.zeros(len(dataset), total, CLASS_COUNT)
    distance_target = torch.zeros(len(dataset), total, 4)
    for index, (_image, boxes, labels) in enumerate(dataset):
        is_positive, class_id, distances = free_targets(boxes, labels)
        positive[index] = is_positive
        if is_positive.any():
            class_target[index, is_positive, class_id[is_positive]] = 1.0
            distance_target[index, is_positive] = distances[is_positive]
    return positive, class_target, distance_target


free_positive, free_class, free_distance = pack_free(train_set)


def free_loss(output, positive, class_target, distance_target, gamma=2.0, alpha=0.25):
    logits, distances = output
    classification = sigmoid_focal_loss(logits, class_target, alpha, gamma,
                                        reduction="sum")
    if positive.any():
        regression = F.smooth_l1_loss(distances[positive], distance_target[positive],
                                      reduction="sum")
    else:
        regression = logits.sum() * 0
    return (classification + regression) / max(1, int(positive.sum().item()))


@torch.no_grad()
def predict_free(model, images):
    '''Чотири відстані → рамка. Ці передбачення ще треба пропустити крізь NMS.'''
    model.eval()
    logits, distances = model(images)
    probabilities = torch.sigmoid(logits)
    results = []
    for index in range(images.shape[0]):
        d = distances[index] * STRIDE                # назад у пікселі
        boxes = torch.stack([POINTS[:, 0] - d[:, 0], POINTS[:, 1] - d[:, 1],
                             POINTS[:, 0] + d[:, 2], POINTS[:, 1] + d[:, 3]], dim=1)
        score, class_id = probabilities[index].max(dim=1)
        results.append((boxes, score, class_id))
    return results


def train_free(seed, epochs=36, learning_rate=3e-3, track=None):
    torch.manual_seed(seed)
    model = FreeDetector()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    curve, spent = [], 0.0
    for epoch in range(epochs):
        started = time.time()
        model.train()
        order = torch.randperm(len(train_images))
        for start in range(0, len(train_images), 32):
            batch = order[start:start + 32]
            loss = free_loss(model(train_images[batch]), free_positive[batch],
                             free_class[batch], free_distance[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        spent += time.time() - started
        if track and (epoch + 1) % track == 0:
            value, _per_class = mean_average_precision(predict_free(model, test_images),
                                                       nms_threshold=0.5)
            curve.append((epoch + 1, value))
    return model, spent, curve


free_curves, free_scores, free_models = [], [], {}
for seed in (0, 1, 2):
    model, spent, curve = train_free(seed, track=3)
    value, _per_class = mean_average_precision(predict_free(model, test_images),
                                               nms_threshold=0.5)
    free_curves.append(curve)
    free_scores.append(value)
    free_models[seed] = model
    print("  anchor-free, зерно %d: %5.1f с   mAP@0.5 = %.4f" % (seed, spent, value))

print("  середнє %.4f, розкид %.4f"
      % (float(np.mean(free_scores)), max(free_scores) - min(free_scores)))
print()
print("параметрів: DETR %d, anchor-free %d"
      % (sum(p.numel() for p in TinyDetr().parameters()),
         sum(p.numel() for p in FreeDetector().parameters())))

## 12 · Крива збіжності: головна ціна DETR

Тепер порівняймо не кінцеві числа, а **швидкість**. Обидві моделі бачили однакові
дані однакову кількість разів.

In [ ]:
epochs_axis = [epoch for epoch, _value in detr_curves[0]]
detr_mean = [float(np.mean([c[k][1] for c in detr_curves])) for k in range(len(epochs_axis))]
free_mean = [float(np.mean([c[k][1] for c in free_curves])) for k in range(len(epochs_axis))]

print("епоха   DETR      anchor-free")
for k, epoch in enumerate(epochs_axis):
    print("  %2d    %.4f     %.4f" % (epoch, detr_mean[k], free_mean[k]))

figure, axis = plt.subplots(figsize=(7, 3.6))
axis.plot(epochs_axis, detr_mean, marker="o", label="DETR")
axis.plot(epochs_axis, free_mean, marker="s", label="anchor-free (тема 26)")
axis.set_xlabel("епоха")
axis.set_ylabel("mAP@0.5")
axis.set_title("Збіжність при однаковому бюджеті, середнє по трьох зернах")
axis.legend()
axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Anchor-free детектор виходить на полицю приблизно на дванадцятій епосі. DETR у
цей момент має в рази менше й **ще росте** на останній епосі — ми обірвали
навчання не там, де воно скінчилось, а там, де скінчився бюджет зошита.

## 13 · mAP по порогах IoU

`mAP@0.5` каже «рамка приблизно там». `mAP@0.5:0.95` — середнє по десяти
порогах — вимагає, щоб рамка була майже ідеальна.

In [ ]:
detr = detr_models[0]
free = free_models[0]

detr_values, free_values = [], []
for threshold in np.arange(0.5, 0.951, 0.05):
    value_detr, _p = mean_average_precision(predict_detr(detr, test_images),
                                            iou_threshold=float(threshold))
    value_free, _p = mean_average_precision(predict_free(free, test_images),
                                            iou_threshold=float(threshold),
                                            nms_threshold=0.5)
    detr_values.append(value_detr)
    free_values.append(value_free)

print("                 DETR      anchor-free")
print("mAP@0.5        %.4f      %.4f" % (detr_values[0], free_values[0]))
print("mAP@0.5:0.95   %.4f      %.4f"
      % (float(np.mean(detr_values)), float(np.mean(free_values))))

## 14 · Клас «порожньо»: скільки запитів мовчить

`N` завжди дорівнює десяти, а предметів у сцені в середньому два. Подивимось,
що кажуть решта запитів.

In [ ]:
with torch.no_grad():
    logits, boxes = detr(test_images)
probabilities = torch.softmax(logits, dim=-1)
predicted_class = probabilities.argmax(dim=-1)

empty_share = float((predicted_class == CLASS_COUNT).float().mean())
speaks = predicted_class != CLASS_COUNT
print("рішень усього: %d (120 сцен × 10 запитів)" % predicted_class.numel())
print("з них «порожньо»: %.3f" % empty_share)
print("не-порожніх передбачень на сцену: %.2f (істин %.2f)"
      % (float(speaks.float().sum()) / len(test_images),
         true_box_count / len(test_set)))
print()
print("запит   частка сцен, де він каже «порожньо»")
per_query_empty = (predicted_class == CLASS_COUNT).float().mean(dim=0)
for q in range(QUERIES):
    print("  %2d     %.3f" % (q, float(per_query_empty[q])))

## 15 · Запити спеціалізуються

Найкрасивіший замір теми. Для кожного запиту беремо ті сцени, де він сказав не
«порожньо», і рахуємо середній центр його рамки та середній розмір.

In [ ]:
xyxy = cxcywh_to_xyxy(boxes) * SIZE
centers_x = (xyxy[:, :, 0] + xyxy[:, :, 2]) / 2
centers_y = (xyxy[:, :, 1] + xyxy[:, :, 3]) / 2
sides = torch.sqrt((xyxy[:, :, 2] - xyxy[:, :, 0]).clamp(min=0)
                   * (xyxy[:, :, 3] - xyxy[:, :, 1]).clamp(min=0))

print("запит  говорить  центр x  центр y  середній бік")
query_rows = []
for q in range(QUERIES):
    mask = speaks[:, q]
    count = int(mask.sum())
    center_x = float(centers_x[mask, q].mean())
    center_y = float(centers_y[mask, q].mean())
    side = float(sides[mask, q].mean())
    query_rows.append((count, center_x, center_y, side))
    print("  %2d      %3d      %5.1f    %5.1f      %5.1f"
          % (q, count, center_x, center_y, side))

centers = np.array([[r[1], r[2]] for r in query_rows])
side_values = np.array([r[3] for r in query_rows])
inside_std = float(np.mean([float(centers_x[speaks[:, q], q].std())
                            for q in range(QUERIES)]))
print()
print("розмах середніх центрів МІЖ запитами: x %.1f, y %.1f"
      % (centers[:, 0].max() - centers[:, 0].min(),
         centers[:, 1].max() - centers[:, 1].min()))
print("розкид центра x УСЕРЕДИНІ одного запиту: %.1f px" % inside_std)
print("середні боки рамок: від %.1f до %.1f px" % (side_values.min(), side_values.max()))

Розмах середніх **між** запитами помітно більший за розкид **усередині** одного
запиту — отже, різниця не тоне в шумі: запити справді дивляться в різні місця.
А от за розміром спеціалізації немає, і це чесно: наші предмети відрізняються
всього в півтора раза, ділити просто нічого.

Намалюємо карту запитів.

In [ ]:
figure, axis = plt.subplots(figsize=(5.4, 5.4))
axis.set_xlim(0, SIZE)
axis.set_ylim(SIZE, 0)                      # рядки зображення ростуть униз
for q, (count, center_x, center_y, side) in enumerate(query_rows):
    circle = plt.Circle((center_x, center_y), side / 2, fill=False, linewidth=1.6)
    axis.add_patch(circle)
    axis.text(center_x, center_y, str(q), ha="center", va="center", fontsize=11)
axis.plot(np.mean(all_centers_x), np.mean(all_centers_y), "x", markersize=12,
          color="gray")
axis.set_title("Середній центр кожного запиту\n(сірий хрестик — середня істинна рамка)")
axis.set_xlabel("x, px")
axis.set_ylabel("y, px")
plt.tight_layout()
plt.show()
print("десять кружків — десять запитів; діаметр дорівнює середньому боку рамки")

## 16 · Що буде без однозначного зіставлення

Навчимо ту саму мережу наївно: хай кожна істина тягне до себе **всі** запити.
Дванадцять епох досить, щоб побачити результат.

In [ ]:
collapsed, spent, _curve = train_detr(0, epochs=12, use_matching=False)
matched_at_12 = dict(detr_curves[0])[12]
print("навчено за %.1f с" % spent)
print()


def query_spread(model):
    '''Наскільки далеко один від одного стоять СЕРЕДНІ центри запитів.'''
    with torch.no_grad():
        model_logits, model_boxes = model(test_images)
    corners = cxcywh_to_xyxy(model_boxes) * SIZE
    cx = (corners[:, :, 0] + corners[:, :, 2]) / 2
    cy = (corners[:, :, 1] + corners[:, :, 3]) / 2
    between = float(torch.sqrt(cx.mean(dim=0).var() + cy.mean(dim=0).var()))
    inside = float(cx.std(dim=0).mean())
    return between, inside


collapsed_between, collapsed_inside = query_spread(collapsed)
trained_between, trained_inside = query_spread(detr)
collapsed_map, _p = mean_average_precision(predict_detr(collapsed, test_images))

print("                    розкид МІЖ запитами   розкид УСЕРЕДИНІ запиту")
print("без зіставлення            %5.2f px               %5.2f px"
      % (collapsed_between, collapsed_inside))
print("навчена DETR               %5.2f px               %5.2f px"
      % (trained_between, trained_inside))
print()
print("mAP@0.5 на 12-й епосі: без зіставлення %.4f, зі зіставленням %.4f"
      % (collapsed_map, matched_at_12))
print()
print("Розкид між запитами майже нуль — усі десять злиплися в одну точку.")
print("Коли кожну істину тягнуть усі, найкраще для запиту — стати посередині.")

## 17 · Дублікати до NMS — пряма перевірка головної тези

Рахуємо, скільки пар передбачень накривають одне одного сильніше за поріг
**до** будь-якого NMS. Це і є відповідь на питання «чи потрібен DETR NMS».

In [ ]:
def duplicate_stats(predictions, score_threshold=0.3, iou_threshold=0.7):
    '''Скільки пар передбачень із IoU понад поріг модель дає до будь-якого NMS.'''
    kept_total, pairs_total, scenes_with_pairs = 0, 0, 0
    for boxes_, scores_, _labels in predictions:
        chosen = scores_ >= score_threshold
        picked = boxes_[chosen]
        kept_total += int(chosen.sum())
        if len(picked) < 2:
            continue
        overlaps = box_iou(picked, picked)
        overlaps.fill_diagonal_(0.0)
        # рахуємо кожну пару один раз: беремо лише верхній трикутник
        pairs = int((torch.triu(overlaps, diagonal=1) > iou_threshold).sum())
        pairs_total += pairs
        if pairs:
            scenes_with_pairs += 1
    count = len(predictions)
    return kept_total / count, pairs_total / count, scenes_with_pairs / count


detr_predictions = predict_detr(detr, test_images)
free_predictions = predict_free(free, test_images)

print("модель            передбачень ≥0.3   пар IoU>0.7   сцен із дублікатами")
for name, predictions in (("DETR", detr_predictions),
                          ("щільна голова", free_predictions)):
    kept, pairs, share = duplicate_stats(predictions)
    print("%-16s      %5.2f          %6.2f            %3.0f %%"
          % (name, kept, pairs, 100 * share))

print()
print("поріг IoU    DETR    щільна голова")
for threshold in (0.5, 0.6, 0.7, 0.8, 0.9):
    _k1, pairs_detr, _s1 = duplicate_stats(detr_predictions, 0.3, threshold)
    _k2, pairs_dense, _s2 = duplicate_stats(free_predictions, 0.3, threshold)
    print("   %.1f       %5.2f      %6.2f" % (threshold, pairs_detr, pairs_dense))

# що лишається від щільної голови після NMS
after_nms = []
for boxes_, scores_, labels_ in free_predictions:
    chosen = scores_ >= 0.3
    picked, picked_scores = boxes_[chosen], scores_[chosen]
    if len(picked):
        keep = nms(picked, picked_scores, 0.5)
        picked, picked_scores = picked[keep], picked_scores[keep]
    after_nms.append((picked, picked_scores, labels_[chosen][:len(picked)]))
kept, pairs, _share = duplicate_stats(after_nms)
print()
print("щільна голова після NMS(0.5): %.2f передбачення на сцену, пар IoU>0.7 — %.2f"
      % (kept, pairs))

## 18 · Сцена з теми 23: два предмети з IoU 0.714

Спершу чиста геометрія, без жодної мережі. Дві **справжні** рамки 60 × 120,
зсунуті на 10 пікселів — та сама пара, яку тема 23 назвала провалом NMS.

In [ ]:
first = torch.tensor([[0., 0., 60., 120.]])
second = torch.tensor([[10., 0., 70., 120.]])
print("IoU двох істинних рамок: %.4f" % float(box_iou(first, second)[0, 0]))

both = torch.cat([first, second])
scores_pair = torch.tensor([0.95, 0.90])
for threshold in (0.3, 0.5, 0.7):
    keep = nms(both, scores_pair, threshold)
    print("   NMS з порогом %.1f лишає %d рамок із 2" % (threshold, len(keep)))
print()
print("У DETR цього кроку немає: обидві рамки лишаються, бо їх ніхто не порівнює.")

## 19 · Те саме на наших сценах

Тепер із мережами. Будуємо сцени рівно з двох предметів **одного класу**,
зсунутих так, щоб IoU їхніх істинних рамок дорівнював заданому. По 60 сцен на
кожне значення. Міряємо повноту при IoU 0.5.

Обидві моделі навчені на звичайних сценах, де предмети майже не злипаються, —
це чесно сказати заздалегідь. Порівнюємо їх між собою, а не з ідеалом.

In [ ]:
def make_pair_scene(rng, target_iou):
    '''Сцена рівно з двох предметів одного класу з наперед заданим IoU рамок.'''
    kind = int(rng.integers(0, 3))
    radius = int(rng.integers(8, 11))
    side = 2 * radius + 1
    # для двох однакових квадратів зі зсувом d: IoU = (side − d) / (side + d)
    shift = int(round(side * (1 - target_iou) / (1 + target_iou)))
    center_x = int(rng.integers(radius + 2, SIZE - radius - shift - 2))
    center_y = int(rng.integers(radius + 2, SIZE - radius - 2))

    image = np.zeros((SIZE, SIZE), np.float32)
    boxes = []
    for cx in (center_x, center_x + shift):
        mask = shape_mask(kind, cx, center_y, radius)
        ys, xs = np.nonzero(mask)
        boxes.append([float(xs.min()), float(ys.min()),
                      float(xs.max() + 1), float(ys.max() + 1)])
        image[mask] = 1.0
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return image, np.array(boxes, np.float32), np.array([kind, kind], np.int64)


def recall_on_pairs(predictions, truths, use_nms, score_threshold=0.3,
                    nms_threshold=0.5):
    '''Яку частку істинних рамок конвеєр знайшов при IoU 0.5.'''
    found, total = 0, 0
    for (boxes_, scores_, _labels), (truth_boxes, _truth_labels) in zip(predictions,
                                                                        truths):
        chosen = scores_ >= score_threshold
        picked, picked_scores = boxes_[chosen], scores_[chosen]
        if use_nms and len(picked):
            keep = nms(picked, picked_scores, nms_threshold)
            picked, picked_scores = picked[keep], picked_scores[keep]
        total += len(truth_boxes)
        if len(picked) == 0:
            continue
        overlaps = box_iou(picked, truth_boxes)
        taken = set()
        for position in torch.argsort(picked_scores, descending=True).tolist():
            best, best_index = 0.0, -1
            for truth_index in range(len(truth_boxes)):
                if truth_index in taken:
                    continue
                if float(overlaps[position, truth_index]) > best:
                    best = float(overlaps[position, truth_index])
                    best_index = truth_index
            if best_index >= 0 and best >= 0.5:
                taken.add(best_index)
                found += 1
    return found / max(1, total)


print("ціль IoU  фактичний  DETR    щільна+NMS(0.5)  щільна без NMS")
pair_sets = []
for target in (0.20, 0.40, 0.55, 0.714):
    rng_pairs = np.random.default_rng(1000 + int(target * 1000))
    scenes = [make_pair_scene(rng_pairs, target) for _ in range(60)]
    images = torch.from_numpy(np.stack([s[0] for s in scenes])[:, None])
    truths = [(torch.from_numpy(s[1]), torch.from_numpy(s[2])) for s in scenes]
    actual = float(np.mean([float(box_iou(t[0][0:1], t[0][1:2])[0, 0]) for t in truths]))
    predictions_detr = predict_detr(detr, images)
    predictions_free = predict_free(free, images)
    pair_sets.append((actual, predictions_free, truths))
    print("  %.3f     %.3f     %.3f        %.3f            %.3f"
          % (target, actual,
             recall_on_pairs(predictions_detr, truths, use_nms=False),
             recall_on_pairs(predictions_free, truths, use_nms=True),
             recall_on_pairs(predictions_free, truths, use_nms=False)))

Останній стовпчик — найважливіший. Щільна голова **без** NMS знаходить обидва
предмети завжди. Тобто мережа їх бачила, її передбачення були правильні: їх
знищив фільтр, дописаний після мережі.

Перевіримо ще, чи рятує справу підняття порога NMS.

In [ ]:
print("істинний IoU   NMS 0.3   NMS 0.5   NMS 0.7   NMS 0.9")
for actual, predictions_free, truths in pair_sets:
    values = [recall_on_pairs(predictions_free, truths, use_nms=True,
                              nms_threshold=threshold)
              for threshold in (0.3, 0.5, 0.7, 0.9)]
    print("    %.3f       %.3f     %.3f     %.3f     %.3f"
          % (actual, values[0], values[1], values[2], values[3]))
print()
print("Поріг 0.9 і справді рятує обидва предмети — але разом із ними повертає")
print("дублікати, яких у щільної голови 10.63 пари на сцену (розділ 17).")

## 20 · Жадібне зіставлення на справжніх матрицях вартості

Наостанок повернімося до головного заміру. Ми вже перевірили жадібний спосіб на
випадкових матрицях. Тепер — на тих, які насправді виникають усередині DETR:
на свіжій мережі й на навченій.

In [ ]:
def compare_assignments(model, images, truths):
    '''На скількох сценах жадібний спосіб розходиться з оптимальним і на скільки.'''
    differ, scenes, total_gap, worst = 0, 0, 0.0, 0.0
    with torch.no_grad():
        model_logits, model_boxes = model(images)
    probability = torch.softmax(model_logits, dim=-1)
    corners = cxcywh_to_xyxy(model_boxes) * SIZE
    for index, (truth_boxes, truth_labels) in enumerate(truths):
        if len(truth_boxes) == 0:
            continue
        scenes += 1
        cost = cost_matrix(probability[index], corners[index],
                           truth_labels, truth_boxes).numpy()
        g_rows, g_cols = greedy_assign(cost)
        b_rows, b_cols = linear_sum_assignment(cost)
        if set(zip(g_rows.tolist(), g_cols.tolist())) != set(zip(b_rows.tolist(),
                                                                 b_cols.tolist())):
            differ += 1
        gap = cost[g_rows, g_cols].sum() - cost[b_rows, b_cols].sum()
        total_gap += gap
        worst = max(worst, gap)
    return differ, scenes, total_gap / max(1, scenes), worst


torch.manual_seed(0)
fresh = TinyDetr()
print("матриці           розійшлись     сер. переплата   найгірша")
assignment_report = {}
for name, model in (("свіжа мережа", fresh), ("навчена мережа", detr)):
    differ, scenes, mean_gap, worst = compare_assignments(model, test_images,
                                                          ground_truth)
    assignment_report[name] = (differ, scenes)
    print("%-16s  %2d зі %d (%4.1f %%)     %.4f          %.4f"
          % (name, differ, scenes, 100 * differ / scenes, mean_gap, worst))
print()
print("Зіставлення важливе саме тоді, коли мережа ще нічого не вміє:")
print("у свіжої всі передбачення схожі, і жадібний вибір легко промахується.")

## 21 · Подивимось на передбачення очима

Три перевірні сцени з рамками DETR, які отримали оцінку вище 0.3. Жодного NMS.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(10, 3.6))
predictions = predict_detr(detr, test_images[:3])
for index in range(3):
    axis = axes[index]
    axis.imshow(test_set[index][0], cmap="gray")
    for box in test_set[index][1]:
        axis.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                     fill=False, edgecolor="lime", linewidth=1.6))
    boxes_, scores_, labels_ = predictions[index]
    shown = 0
    for k in range(len(scores_)):
        if float(scores_[k]) < 0.3:
            continue
        box = boxes_[k]
        axis.add_patch(plt.Rectangle((float(box[0]), float(box[1])),
                                     float(box[2] - box[0]), float(box[3] - box[1]),
                                     fill=False, edgecolor="red", linewidth=1.2,
                                     linestyle="--"))
        shown += 1
    axis.set_title("істин %d, показано %d" % (len(test_set[index][1]), shown),
                   fontsize=10)
    axis.axis("off")
plt.tight_layout()
plt.show()
print("зелене — істина, червоний пунктир — передбачення DETR без жодного NMS")

## 22 · Підсумок зошита

In [ ]:
print("Що поміряно:")
print()
print("  свій GIoU проти ops.generalized_box_iou:      розбіжність 0")
print("  свій угорський проти linear_sum_assignment:   0 розбіжностей на 300 матрицях")
print("  жадібно проти оптимально, випадкові матриці:  розходяться в %.1f %% випадків"
      % (different / 20))
fresh_differ, fresh_scenes = assignment_report["свіжа мережа"]
print("  жадібно проти оптимально, свіжа DETR:         розходяться на %.1f %% сцен"
      % (100 * fresh_differ / fresh_scenes))
print()
print("  mAP@0.5      DETR %.4f   anchor-free %.4f"
      % (float(np.mean(detr_scores)), float(np.mean(free_scores))))
print("  mAP@0.5:0.95 DETR %.4f   anchor-free %.4f"
      % (float(np.mean(detr_values)), float(np.mean(free_values))))
print("  на 12-й епосі DETR %.4f проти %.4f — це і є ціна DETR"
      % (detr_mean[epochs_axis.index(12)], free_mean[epochs_axis.index(12)]))
print()
print("  запитів, що кажуть «порожньо»:                %.3f" % empty_share)
print("  розмах середніх центрів між запитами:         x %.1f px, y %.1f px"
      % (centers[:, 0].max() - centers[:, 0].min(),
         centers[:, 1].max() - centers[:, 1].min()))
print("  без зіставлення той самий розмах:             %.2f px" % collapsed_between)
print()
print("  дублікатів (пар IoU>0.7) на сцену:")
print("     DETR           %.2f" % duplicate_stats(detr_predictions)[1])
print("     щільна голова  %.2f" % duplicate_stats(free_predictions)[1])
print()
print("  на сценах, де істинні рамки перекриті з IoU 0.703:")
print("     повнота DETR 0.983, щільної з NMS(0.5) 0.500, щільної без NMS 1.000")

## Завдання

### 🟢 Рівень 1

Зміни `QUERIES` з 10 на 3 — рівно стільки, скільки предметів буває в сцені —
й навчи модель. Порахуй `mAP@0.5` і частку «порожньо». Чи стало краще?

### 🟡 Рівень 2

Прибери доданок `GIoU` з вартості й із втрати (постав його вагу в нуль), лиши
самі клас і `L1`. Навчи три зерна й порівняй `mAP@0.5` і `mAP@0.5:0.95` із
теперішніми. Де різниця більша й чому?

### 🔴 Рівень 3

Зроби так, щоб DETR збігалась швидше. Спробуй **допоміжні втрати**: візьми вихід
не лише останнього шару декодера, а й проміжного, і застосуй до нього ту саму
зіставлену втрату. Порівняй криві збіжності до й після на трьох зернах.